<a href="https://colab.research.google.com/github/researchbas/javanese_word_segmentation/blob/main/Javanese_Dataset_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset



In [ ]:
aks = [
    'ꦄ','ꦅ', 'ꦆ', 'ꦇ', 'ꦈ', 'ꦉ', 'ꦊ', 'ꦋ', 'ꦌ', 'ꦍ', 'ꦎ', 'ꦏ',
    'ꦐ', 'ꦑ', 'ꦒ', 'ꦓ', 'ꦔ', 'ꦕ', 'ꦖ', 'ꦗ', 'ꦘ', 'ꦙ', 'ꦚ', 'ꦛ', 'ꦜ','ꦝ', 'ꦞ', 'ꦟ',
    'ꦠ', 'ꦡ', 'ꦢ', 'ꦣ', 'ꦤ', 'ꦥ', 'ꦦ', 'ꦧ', 'ꦨ', 'ꦩ', 'ꦪ', 'ꦫ', 'ꦬ', 'ꦭ', 'ꦮ', 'ꦯ',
    'ꦰ', 'ꦱ', 'ꦲ'
]

def cluster(word):
    word = word.replace('-', '')
    jcc = ""
    jcc_list = []
    label_list = [1]
    flag = False
    for index, char in enumerate(word):
        jcc = jcc + char
        if index < len(word) - 1:  # Check if it's not the last character
            next_character = word[index + 1]
            if next_character in aks:
                jcc_list.append(jcc)
                label_list.append(0)
                jcc = ""
        else:
            jcc_list.append(jcc)
            jcc = ""

    return jcc_list, label_list

In [ ]:
def preprocessData(data):
    X = []
    Y = []
    for text in data:
        jcc_list = []
        label_list = []
        raw = text.split("|")
        for word in raw:
            jcc, label = cluster(word)
            jcc_list.extend(jcc)
            label_list.extend(label)

        X.append(jcc_list)
        Y.append(label_list)
    return X, Y

In [ ]:
expected_chars = 'ꦀꦁꦂꦃꦄꦅꦆꦇꦈꦉꦊꦋꦌꦍꦎꦏꦐꦑꦒꦓꦔꦕꦖꦗꦘꦙꦚꦛꦜꦝꦞꦟꦠꦡꦢꦣꦤꦥꦦꦧꦨꦩꦪꦫꦬꦭꦮꦯꦰꦱꦲ꦳ꦴꦵꦶꦷꦸꦹꦺꦻꦼꦽꦾꦿ꧀'
char_to_index = {char: idx for idx, char in enumerate(expected_chars)}
NUM_CHARS = len(char_to_index)

# Function to encode a character cluster
def multi_hot_encode_chars(chars):
    # print(chars)
    zero_vector = np.zeros(NUM_CHARS)
    # If the chars has fewer than max_length characters, pad with zero vectors
    for char in chars:
        zero_vector[char_to_index[char]] = 1
    return zero_vector

# Function to pad the sequences
def pad_sequences(dataset, max_sequence_length=None):
    padded_sequences = []
    if max_sequence_length is None:
        max_sequence_length = max(len(sequence) for sequence in dataset)
        print(max_sequence_length)
    for sequence in dataset:
        # Pad the sequence with zero vectors if it's shorter than max_sequence_length
        while len(sequence) < max_sequence_length:
            sequence.append(np.zeros(NUM_CHARS))  # Pad with zero vector of shape (130,)
        padded_sequences.append(sequence)
    return padded_sequences

# Function to pad labels
def pad_labels(labels, max_len=None, padding_value=-1):
    if max_len is None:
        max_len = max(len(label) for label in labels)  # Find the maximum length in the labels
    padded_labels = np.full((len(labels), max_len), padding_value)  # Assuming binary labels
    for i, label in enumerate(labels):
        padded_labels[i, :len(label)] = label  # Fill the padded array
    return padded_labels

In [ ]:
import copy

# Define model
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(BiLSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, 1)  # Bidirectional doubles the hidden size

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # LSTM output
        logits = self.fc(lstm_out)  # Fully connected layer output
        logits = logits.squeeze(-1)  # Remove last dimension to match the shape of labels
        return logits

# Training loop with early stopping (convergence criterion)
def train_model(model, train_dataset, val_dataset, num_epochs=100, batch_size=64,
                padding_value=-1, patience=5, min_delta=1e-3):
    """
    Convergence criterion: training stops when the validation loss has not
    improved by more than `min_delta` for `patience` consecutive epochs.
    The weights of the best epoch are restored at the end.
    """
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss(reduction='none')  # Binary cross-entropy with logits
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Lists to store metrics per epoch
    train_losses, val_losses = [], []
    precisions, recalls, f1_scores = [], [], []

    # Early-stopping state
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0
    converged = False

    for epoch in range(num_epochs):
        total_train_loss = 0.0
        model.train()  # Set model to training mode
        for X_batch, Y_batch in train_dataloader:
            optimizer.zero_grad()  # Zero out gradients

            outputs = model(X_batch)  # Forward pass

            # Mask where Y_batch is not the padding value
            mask = (Y_batch != padding_value).float()

            loss = criterion(outputs, Y_batch)
            masked_loss = loss * mask

            # Average the loss only over non-padded elements
            if mask.sum() > 0:
                masked_loss = masked_loss.sum() / mask.sum()
            else:
                masked_loss = masked_loss.sum()

            masked_loss.backward()  # Backpropagation
            optimizer.step()  # Update the model parameters

            total_train_loss += masked_loss.item()  # Accumulate loss

        # Validation phase
        model.eval()  # Set model to evaluation mode
        total_val_loss = 0.0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X_val, Y_val in val_dataloader:
                val_outputs = model(X_val)  # Forward pass
                val_mask = (Y_val != padding_value).float()
                val_loss = criterion(val_outputs, Y_val)
                masked_val_loss = val_loss * val_mask
                if val_mask.sum() > 0:
                    masked_val_loss = masked_val_loss.sum() / val_mask.sum()
                total_val_loss += masked_val_loss.item()

                # --- Collect for metrics ---
                preds = torch.sigmoid(val_outputs)  # Convert logits to probabilities
                preds = (preds > 0.5).float()       # Threshold to binary

                # Apply mask (ignore padding)
                valid_indices = (Y_val != padding_value)
                valid_preds = preds[valid_indices].cpu().numpy()
                valid_labels = Y_val[valid_indices].cpu().numpy()

                all_preds.extend(valid_preds)
                all_labels.extend(valid_labels)

        # --- Compute metrics ---
        if len(all_labels) > 0:
            precision = precision_score(all_labels, all_preds, zero_division=0)
            recall = recall_score(all_labels, all_preds, zero_division=0)
            f1 = f1_score(all_labels, all_preds, zero_division=0)
        else:
            precision = recall = f1 = 0.0

        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_val_loss = total_val_loss / len(val_dataloader)

        # Store metrics
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

        print(
            f"Epoch [{epoch+1}/{num_epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}"
        )

        # --- Convergence check (early stopping on validation loss) ---
        if best_val_loss - avg_val_loss > min_delta:
            best_val_loss = avg_val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                converged = True
                print(f"Converged: validation loss did not improve by more than "
                      f"{min_delta} for {patience} consecutive epochs. "
                      f"Stopped at epoch {epoch+1}; best epoch = {best_epoch}.")
                break

    if not converged:
        print(f"Convergence criterion NOT met within {num_epochs} epochs "
              f"(best epoch = {best_epoch}).")

    # Restore the weights of the best epoch
    if best_state is not None:
        model.load_state_dict(best_state)

    # Return model + metric logs
    return model, {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "val_f1": f1_scores,
        "val_precision": precisions,
        "val_recall": recalls,
        "best_epoch": best_epoch,
        "stopped_epoch": epoch + 1,
        "converged": converged,
    }

INPUT_SIZE = NUM_CHARS  # Size of the multi-hot encoded vector (65)
HIDDEN_SIZE = 128

In [ ]:
##load the dataset here
df = pd.read_excel('javanese_word_segmentation.xlsx', sheet_name='Sheet1')
dataset = df['segmented_text'].head(11000).tolist()
X, Y = preprocessData(dataset)

# # Step 1: Split into train + temp (where temp will be split into validation and test)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.1, random_state=42)

X_train_encoded = [[multi_hot_encode_chars(jcc) for jcc in jcc_group] for jcc_group in X_train]
X_val_encoded = [[multi_hot_encode_chars(jcc) for jcc in jcc_group] for jcc_group in X_val]

# # Pad the training and val sequences separately
X_train_padded = pad_sequences(X_train_encoded, max(len(seq) for seq in X_train))
X_val_padded = pad_sequences(X_val_encoded, max(len(seq) for seq in X_val))

# # Convert the padded sequences to NumPy arrays
X_train_padded_np = np.array(X_train_padded)
X_val_padded_np = np.array(X_val_padded)

# Pad the training and val labels
Y_train_padded = pad_labels(Y_train)
Y_val_padded = pad_labels(Y_val)

Y_train_tensor = torch.tensor(Y_train_padded, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val_padded, dtype=torch.float32)

X_train_tensor = (torch.tensor(X_train_padded_np, dtype=torch.float32))
X_val_tensor = (torch.tensor(X_val_padded_np, dtype=torch.float32))

In [ ]:
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

model_lstm = BiLSTMModel(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE)
# Train the model
model, history = train_model(model_lstm, train_dataset, val_dataset)



Epoch [1/20] | Train Loss: 0.4852 | Val Loss: 0.3078 | Precision: 0.8004 | Recall: 0.7887 | F1: 0.7945
Epoch [2/20] | Train Loss: 0.2703 | Val Loss: 0.2362 | Precision: 0.8772 | Recall: 0.8142 | F1: 0.8445
Epoch [3/20] | Train Loss: 0.2239 | Val Loss: 0.2077 | Precision: 0.8609 | Recall: 0.8860 | F1: 0.8733
Epoch [4/20] | Train Loss: 0.1959 | Val Loss: 0.1806 | Precision: 0.8872 | Recall: 0.8937 | F1: 0.8904
Epoch [5/20] | Train Loss: 0.1739 | Val Loss: 0.1629 | Precision: 0.9085 | Recall: 0.8924 | F1: 0.9004
Epoch [6/20] | Train Loss: 0.1555 | Val Loss: 0.1517 | Precision: 0.8980 | Recall: 0.9238 | F1: 0.9107
Epoch [7/20] | Train Loss: 0.1404 | Val Loss: 0.1395 | Precision: 0.9385 | Recall: 0.8883 | F1: 0.9127
Epoch [8/20] | Train Loss: 0.1272 | Val Loss: 0.1246 | Precision: 0.9167 | Recall: 0.9384 | F1: 0.9274
Epoch [9/20] | Train Loss: 0.1163 | Val Loss: 0.1217 | Precision: 0.9348 | Recall: 0.9218 | F1: 0.9282
Epoch [10/20] | Train Loss: 0.1077 | Val Loss: 0.1194 | Precision: 0.9460